# Bước 1: tạo bản sao video EK100 ở đúng độ phân giải của eval (chỉ cần CPU)

Video gốc quá lớn để lưu lại (val 184 GB, train khoảng 1 TB). Notebook này tải từng video gốc, **resize giống hệt pipeline
V-JEPA 2** (cv2 `INTER_LINEAR`, cạnh ngắn 292; pipeline sẽ bỏ qua bước resize khi gặp ảnh 292), pad chiều rộng 519 lên 520
(vùng center crop không đổi), rồi nén x264 `yuv444p` crf 12. Tất cả frame và đúng fps gốc được giữ nguyên, nên frame id
trong annotation vẫn trỏ đúng frame. Mỗi file được kiểm tra số frame và fps so với bản gốc (ghi trong `manifest.csv`).

**Cài đặt:** Accelerator **None** (CPU, không tốn quota GPU), Internet **On**.

**Chia việc** (mỗi phiên tối đa khoảng 11 giờ và 19 GB output; nhiều tài khoản chạy song song được):

| SPLIT | NUM_SHARDS | Ghi chú |
|---|---|---|
| `val` | 2 | 138 video, ~13 giờ video. Nên làm trước, vì cần để kiểm tra chất lượng bản sao |
| `train` | 8 | 495 video, ~75 giờ video, ~1 TB tải về |

Mỗi phiên chạy một `SHARD_ID` (0 … NUM_SHARDS−1). Nếu phiên dừng vì hết giờ hoặc hết dung lượng: *Add Input* output của phiên đó
rồi chạy lại cùng `SHARD_ID`, các video đã xong sẽ được bỏ qua. Xong thì tạo **Dataset từ Output** (tab Output, *New Dataset*),
ví dụ `ek100-292-val-0`. Sau này chỉ cần *Add Input* tất cả các dataset đó, code tự tìm video trong `/kaggle/input`.

In [ ]:
SPLIT = "val"        # "val" hoặc "train"
SHARD_ID = 0
NUM_SHARDS = 2       # val: 2, train: 8

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!apt-get -qq install -y aria2 ffmpeg > /dev/null 2>&1; pip install -q decord av imageio-ffmpeg
R = "/kaggle/working/repo"
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!nproc; df -h /kaggle/tmp /kaggle/working | tail -2

In [ ]:
!python $R/scripts/reencode_ek100.py --split $SPLIT --shard_id $SHARD_ID --num_shards $NUM_SHARDS \
    --annotations_dir /kaggle/working/ek100-ann --out_dir /kaggle/working/ek100_292 --tmp_dir /kaggle/tmp/raw \
    --encode_workers 2 --ffmpeg_threads 2 --parallel_downloads 2 --max_hours 11 --max_output_gb 19

In [ ]:
import pandas as pd
m = pd.read_csv("/kaggle/working/ek100_292/manifest.csv")
print(m.ok.value_counts(), "\nmedian bitrate (Mbit/s):", m.mbps.median(), "\n", m[~m.ok])
!du -sh /kaggle/working/ek100_292